<a href="https://colab.research.google.com/github/LennartRedlich/Capstone-Project-/blob/Anh/src/notebooks/Feature_engineering_Seniority.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Feature engineering and conventional machine learning - Seniority**
In this approach, we focus on feature engineering combined with conventional machine learning models. Instead of relying primarily on textual representations, meaningful structured features are derived from the LinkedIn CV data, such as career history indicators and categorical job attributes. These features are then used to train standard classification models to predict seniority. This approach aims to improve interpretability and robustness by leveraging domain knowledge and structured career information.

## 1. Preparing Data Set


In [ ]:
!pip install category_encoders
!pip install catboost
import json
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
import category_encoders as ce
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score

In [ ]:

CURRENT_YEAR = datetime.now().year

with open("/content/linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs = []

def extract_year(date_str):
    if not date_str:
        return None
    try:
        return int(date_str[:4])
    except:
        return None

for person_id, cv in enumerate(cvs):

    # 1. find previous jobs
    num_prev_jobs = sum(
        1 for job_item in cv if job_item.get("status") != "ACTIVE"
    )

    # 2. find the individual's start working time
    start_years = []
    for job_item in cv:
        year = extract_year(job_item.get("startDate"))
        if year:
            start_years.append(year)

    first_year = min(start_years) if start_years else None

    # 3. label ACTIVE job
    for job in cv:
        if job.get("status") == "ACTIVE":

            total_years_experience = None
            if first_year:
                total_years_experience = CURRENT_YEAR - first_year

            jobs.append({
                **job,
                "person_id": person_id,
                "num_previous_jobs": num_prev_jobs,
                "total_years_experience": total_years_experience
            })

df_active = pd.DataFrame(jobs)



In [ ]:
df_active.head(10)

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id,num_previous_jobs,total_years_experience
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management,0,1,26
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management,0,1,26
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional,0,1,26
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management,0,1,26
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management,0,1,26
5,Computer Solutions,https://www.linkedin.com/company/computer-solu...,Solutions Architect,2024-03,None,ACTIVE,Information Technology,Professional,1,7,24
6,Udo Weber,https://www.linkedin.com/company/udo-weber,Medizintechnik Beratung,2025-01,None,ACTIVE,Consulting,Professional,2,2,39
7,Grupo Viajes Kontiki.,,Director expansión de negocio.,2024-09,None,ACTIVE,Business Development,Director,3,4,16
8,Air & Ground Operations Consultancy,https://www.linkedin.com/company/agoc-spain,Gerente comercial,2024-04,None,ACTIVE,Sales,Lead,3,4,16
9,Viajes Oceano S.L.,,Administrador Unico,2010-12,None,ACTIVE,Administrative,Professional,3,4,16


The features "num_previous_jobs" and "total_years_experience" were introduced to capture an individual’s career progression beyond the current job title.
- The number of previous jobs serves as a proxy for career mobility and accumulated professional exposure, which is often correlated with seniority.
- Similarly, total years of experience directly reflect the length of time an individual has spent in the workforce, a key indicator of professional maturity.

These features provide quantitative signals that complement textual information from job titles. By incorporating them, the model can better distinguish between early-career and senior professionals, even when job titles alone are ambiguous.

## 2. Endcoding the seniority and spliting the data set
Initially, the job title (position) and department (department) features were encoded using Count Encoding, which represents each category by its frequency in the dataset. This simple encoding allowed the models to capture basic distributional information while keeping the feature space manageable.

In [ ]:
#encode the sennority
seniority_order = [["Professional","Junior", "Senior", "Lead", "Management", "Director"]]

# Initialize the OrdinalEncoder with the defined order
encoder = OrdinalEncoder(categories=seniority_order)

# Fit and transform the 'seniority' column to create the encoded column
df_active.loc[:, 'seniority_encoded'] = encoder.fit_transform(df_active[['seniority']])

In [ ]:
X = df_active[["position","department","num_previous_jobs","total_years_experience"]]
y = df_active["seniority_encoded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Encoding categorical variables

In [ ]:


categorical_cols = ['position','department']
numeric_cols = ['num_previous_jobs','total_years_experience']

categorical_transformer = ce.CountEncoder(cols=categorical_cols)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_cols)
    ], remainder='passthrough'
)



##4. Training the model

 The three models—Logistic Regression, Random Forest, and CatBoost—were chosen to cover a range of modeling approaches suitable for categorical data.
 - Logistic Regression serves as a simple linear baseline that is easy to interpret and can reveal whether the features contain strong linear signals.
 - Random Forest captures non-linear interactions and provides robustness against noise, making it suitable for heterogeneous CV data.
 - CatBoost was included for its ability to handle categorical features natively and perform well even with small to medium-sized datasets, which is typical for LinkedIn CVs. Together, these models offer a balanced perspective on model performance before applying more advanced feature engineering.



In [ ]:

#Define models
random_forest = RandomForestClassifier(n_estimators=200, random_state=42)
CatBoost = CatBoostClassifier(
            iterations=300,
            learning_rate=0.1,
            depth=6,
            loss_function="MultiClass",
            verbose=False,
            random_seed=42
        )
Logistic_Regression = LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
            solver='liblinear'
        )

# Train and evaluate
for model in [random_forest, CatBoost,Logistic_Regression]:
  complete_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)])
  complete_pipeline.fit(X_train, y_train)
  y_pred = complete_pipeline.predict(X_test)
  acc = accuracy_score(y_test, y_pred)
  print(f"{type(model).__name__} accuracy: {acc:.4f}")

RandomForestClassifier accuracy: 0.4320
CatBoostClassifier accuracy: 0.4160
LogisticRegression accuracy: 0.4320


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(


- All three models—Logistic Regression, Random Forest, and CatBoost—achieved relatively low accuracy, staying around 40%. This indicates that the encoded features did not adequately represent the information needed to distinguish different seniority levels. In particular, job titles with similar seniority meanings but different wording were treated as separate categories, which added noise rather than useful signal. As a result, the models showed limited ability to generalize. These findings highlighted the need for more meaningful feature transformations and motivated the subsequent model improvement steps.

## 5. Improving the model

### a) Mapping title & one-hot encoder
To improve model performance, job titles would be mapped to an ordinal position level representing career hierarchy and onehot encoder would be applied for catagorical feature "department". This transformation reduces noise, encodes domain knowledge, and aligns better with the ordinal nature of seniority labels, resulting in more stable and interpretable models.

In [ ]:
position_map = {
    "intern": 0,
    "trainee": 0,
    "junior": 1,
    "associate": 1,
    "analyst": 2,
    "senior": 3,
    "lead": 4,
    "manager": 4,
    "head": 5,
    "director": 6,
    "vp": 7,
    "ceo": 8
}

df_active["position_level"] = (
    df_active["position"]
    .str.lower()
    .map(lambda x: next(
        (v for k, v in position_map.items() if k in x),
        2
    )))


In [ ]:
X_mt = df_active[["position_level","department","num_previous_jobs","total_years_experience"]]
y = df_active["seniority_encoded"]

X_train_mt, X_test_mt, y_train_mt, y_test_mt = train_test_split(X_mt, y, test_size=0.2, random_state=42)

preprocessor_mt = ColumnTransformer(
    transformers=[
        ("dept", OneHotEncoder(handle_unknown="ignore"), ["department"])
    ],
    remainder="passthrough"
)

#training
for model in [random_forest, CatBoost,Logistic_Regression]:
  complete_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_mt),
    ('model', model)])
  complete_pipeline.fit(X_train_mt, y_train_mt)
  y_pred_mt = complete_pipeline.predict(X_test_mt)
  acc = accuracy_score(y_test_mt, y_pred_mt)
  print(f"{type(model).__name__} accuracy: {acc:.4f}")

RandomForestClassifier accuracy: 0.5200
CatBoostClassifier accuracy: 0.5760
LogisticRegression accuracy: 0.4720


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(


- After applying feature engineering, the model performance improved noticeably. Mapping job titles to ordinal position levels helped capture the hierarchical structure inherent in seniority, reducing noise from highly variable raw titles As a result, Logistic Regression achieved an accuracy of around 0.47, while tree-based models reached approximately 0.5, slightly increase from the baseline.

### b) Rule-based matching + One-hot encoder
Rule-based matching provides a simple yet effective way to leverage domain knowledge by directly linking specific keywords in job titles to seniority levels. This approach is particularly useful when the dataset is small or when certain titles strongly indicate a seniority category. By applying rule-based rules in combination with feature engineering - one-hot encoder, the model can benefit from high-precision signals that are difficult to learn automatically

In [ ]:
#rule-based matching

df_seniority = pd.read_csv("seniority-v2.csv")

seniority_dict = (
    df_seniority
    .groupby("label")["text"]
    .apply(list)
    .to_dict()
)
def predict_seniority(sen, seniority_dict):
    sen = sen.lower()

    for label, texts in seniority_dict.items():
        for t in texts:
            if t.lower() in sen:
                return label

    return "Other"

predictions = []

for sen in df_active["position"]:
    pred = predict_seniority(sen, seniority_dict)
    predictions.append(pred)

df_active["predicted_seniority"] = predictions

In [ ]:
#split dataset
X_rb = df_active[["predicted_seniority","department","num_previous_jobs","total_years_experience"]]
y_rb = df_active["seniority_encoded"]

X_train_rb, X_test_rb, y_train_rb, y_test_rb = train_test_split(X_rb, y_rb, test_size=0.2, random_state=42)

#Encode the features
preprocessor_rb = ColumnTransformer(
    transformers=[
        ("dept", OneHotEncoder(handle_unknown="ignore"), ["department"]),
        ("pos", OneHotEncoder(handle_unknown="ignore"), ["predicted_seniority"])
    ],
    remainder="passthrough"
)

#training
for model in [random_forest, CatBoost,Logistic_Regression]:
  complete_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_rb),
    ('model', model)])
  complete_pipeline.fit(X_train_rb, y_train_rb)
  y_pred_rb = complete_pipeline.predict(X_test_rb)
  acc = accuracy_score(y_test_rb, y_pred_rb)
  print(f"{type(model).__name__} accuracy: {acc:.4f}")

RandomForestClassifier accuracy: 0.6560
CatBoostClassifier accuracy: 0.6640
LogisticRegression accuracy: 0.6400


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(


- After incorporating rule-based matching and applying one-hot encoding to the categorical features, the model performance improved significantly. All three models reached an accuracy of around 65%, indicating that the additional domain knowledge helped capture seniority-related patterns more effectively. Among the three models, CatBoost achieved slightly better performance, reflecting its strength in handling categorical features and complex interactions.
- The rule-based rules provided clear and high-precision signals, especially for job titles strongly associated with specific seniority levels.
- One-hot encoding further allowed the models to distinguish categorical information without introducing frequency-related noise.